# Ultimate Model: Advanced Feature Engineering & LightGBM

Mit unserem letzten Modell haben wir die **0.7735** auf dem Kaggle Leaderboard erreicht. Jetzt greifen wir nach den Sternen (0.80+).
Wir erweitern unser Sub-Window Feature Engineering um die mächtigsten Signal-Verarbeitungs-Features:
1. **Frequenzbereich (FFT)**: Wie stark schwingt das Signal? (Erkennt z.B. wiederkehrendes Auftreten wie bei Schritten).
2. **Korrelation**: Bewegen sich X, Y und Z-Achse gemeinsam oder unabhängig?
3. **Signal-Energie & Derivate**: Wie viel rohe "Kraft" steckt in der Bewegung und wie ruckartig ist sie?

In [ ]:
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
from scipy.stats import skew, kurtosis
import warnings
warnings.filterwarnings('ignore')

## 1. Daten laden (.npz)

In [ ]:
KAGGLE_PATH = '/kaggle/input/datasets/axxtur/nycu-data-mining-assignment-3'
if not os.path.exists(KAGGLE_PATH):
    KAGGLE_PATH = '/kaggle/input/nycu-data-mining-assignment-3'
    if not os.path.exists(KAGGLE_PATH):
        KAGGLE_PATH = 'nycu-data-mining-assignment-3'

print("Lade Train Data...")
train_data = np.load(os.path.join(KAGGLE_PATH, 'train_data.npz'), allow_pickle=True)
X_train_raw = train_data['X']
y_train = train_data['y']
file_ids_train = train_data['file_ids']
user_ids_train = train_data['user_ids']

print("Lade Test Data...")
test_data = np.load(os.path.join(KAGGLE_PATH, 'test_data.npz'), allow_pickle=True)
X_test_raw = test_data['X']
file_ids_test = test_data['file_ids']
user_ids_test = test_data['user_ids']

## 2. Advanced Feature Extraction (Globale + Sub-Window + Signal Processing)
Wir nutzen Numpy-Magie, um Frequenz (FFT), Energie und Korrelation in Mikrosekunden zu berechnen.

In [ ]:
def extract_features_np(X):
    def calc_stats(arr, axis):
        f = []
        # 1. Standard Stats
        f.append(np.mean(arr, axis=axis))
        f.append(np.std(arr, axis=axis))
        f.append(np.min(arr, axis=axis))
        f.append(np.max(arr, axis=axis))
        f.append(np.median(arr, axis=axis))
        f.append(np.percentile(arr, 25, axis=axis))
        f.append(np.percentile(arr, 75, axis=axis))
        f.append(skew(arr, axis=axis))
        f.append(kurtosis(arr, axis=axis))
        f.append(np.max(arr, axis=axis) - np.min(arr, axis=axis)) # Range
        
        # 2. Energie (Summe der Quadrate)
        f.append(np.sum(arr**2, axis=axis))
        
        # 3. Ruckartigkeit (Derivate / Diffs)
        diffs = np.diff(arr, axis=axis)
        f.append(np.mean(np.abs(diffs), axis=axis)) # Mean Absolute Difference
        f.append(np.std(diffs, axis=axis))
        
        # 4. Frequenz-Features (Fast Fourier Transform)
        fft_vals = np.abs(np.fft.rfft(arr, axis=axis))
        f.append(np.mean(fft_vals, axis=axis))
        f.append(np.std(fft_vals, axis=axis))
        f.append(np.max(fft_vals, axis=axis))
        
        # 5. Korrelation zwischen Achsen (Nur für die ersten 3 Kanäle: mean_x, mean_y, mean_z)
        def calc_corr(a, b, ax):
            a_mean = np.mean(a, axis=ax, keepdims=True)
            b_mean = np.mean(b, axis=ax, keepdims=True)
            a_std = np.std(a, axis=ax, keepdims=True)
            b_std = np.std(b, axis=ax, keepdims=True)
            cov = np.mean((a - a_mean) * (b - b_mean), axis=ax, keepdims=True)
            return np.squeeze(cov / (a_std * b_std + 1e-8), axis=ax)
            
        f.append(calc_corr(arr[..., 0], arr[..., 1], axis)) # Korrelation X-Y
        f.append(calc_corr(arr[..., 0], arr[..., 2], axis)) # Korrelation X-Z
        f.append(calc_corr(arr[..., 1], arr[..., 2], axis)) # Korrelation Y-Z
        
        # 6. Magnitude
        mag = np.sqrt(arr[..., 0]**2 + arr[..., 1]**2 + arr[..., 2]**2)
        f.append(np.mean(mag, axis=axis, keepdims=True))
        f.append(np.std(mag, axis=axis, keepdims=True))
        f.append(np.max(mag, axis=axis, keepdims=True))
        return f

    # Globale Features (über alle 300 Sekunden)
    global_feats = calc_stats(X, axis=1)
    global_feats = [f.reshape(X.shape[0], -1) for f in global_feats]
    
    # Sub-Window Features (5 Blöcke à 60 Sekunden)
    X_sub = X.reshape(X.shape[0], 5, 60, 6)
    sub_feats = calc_stats(X_sub, axis=2)
    sub_feats = [f.reshape(X.shape[0], -1) for f in sub_feats]
    
    # Zusammenführen
    all_features = np.concatenate(global_feats + sub_feats, axis=1)
    return all_features

print("Extrahiere Features...")
X_train_feat = extract_features_np(X_train_raw)
X_test_feat = extract_features_np(X_test_raw)

print(f"Anzahl der generierten Features pro Sample: {X_train_feat.shape[1]}")

## 3. Modell Training (Cross-Validation)

In [ ]:
gkf = GroupKFold(n_splits=5)
models = []
scores = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train_feat, y_train, groups=user_ids_train)):
    X_tr, X_va = X_train_feat[train_idx], X_train_feat[val_idx]
    y_tr, y_va = y_train[train_idx], y_train[val_idx]
    
    # Stärkeres LightGBM Modell für die vielen neuen Features
    clf = lgb.LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.01,
        random_state=42,
        class_weight='balanced',
        n_jobs=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        max_depth=7,
        num_leaves=64
    )
    
    clf.fit(
        X_tr, y_tr, 
        eval_set=[(X_va, y_va)], 
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    val_preds = clf.predict(X_va)
    fold_f1 = f1_score(y_va, val_preds, average='macro')
    scores.append(fold_f1)
    models.append(clf)
    
    print(f"Fold {fold+1} F1-Macro: {fold_f1:.4f}")

print(f"\nOverall Cross-Validation F1-Macro: {np.mean(scores):.4f}")

## 4. Test Predictions und Kaggle Submission

In [ ]:
test_preds_proba = np.zeros((len(X_test_feat), 6))

for clf in models:
    test_preds_proba += clf.predict_proba(X_test_feat) / len(models)

final_preds = np.argmax(test_preds_proba, axis=1)

submission = pd.DataFrame({
    'Id': file_ids_test,
    'Label': final_preds
})

submission.to_csv('submission_ultimate.csv', index=False)
print("Saved submission_ultimate.csv! Ready for Kaggle upload.")
submission.head()